# CountVideoBench — Exact Match by object class

This notebook uses the modules of this folder (`config.py`, `data.py`, `metrics.py`, `plots.py`) and produces (Appendix F of the paper):

- `output/heatmap_accuracy_by_category.pdf` — Figure 17: Exact Match by model, category and ground-truth count.
- `output/heatmap_accuracy_by_category.csv` — the values of Figure 17.
- A **pandas DataFrame** with the column hierarchy of Table 6 (`Model`, `Shape → Cube / Sphere`, `Color → Red / Green / Blue / Yellow / Purple`).
- `output/table_accuracy_by_category.csv` — Table 6, this DataFrame as CSV.

All Exact Match scores are computed from `../paper_results/*.csv` (counts used in the paper). Missing or non-numerical answers count as incorrect. The table values are percentages, rounded to the nearest integer for display.

## 1. Setup and load predictions

In [ ]:
from pathlib import Path
from collections.abc import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

from config import DEFAULT_INPUT_PATTERN, DEFAULT_OUTPUT_DIR, apply_matplotlib_style
from data import (
    load_model_csvs, preprocess,
    COL_ANSWER, COL_COLOR, COL_GT, COL_MODEL, COL_SHAPE,
)
from metrics import _exact_match_mask, exact_match_by_visual_attributes
from plots import _save_and_show

apply_matplotlib_style()
OUTPUT_DIR = Path(DEFAULT_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_raw, raw_models = load_model_csvs(DEFAULT_INPUT_PATTERN)
df = preprocess(df_raw)

model_order = sorted(df[COL_MODEL].dropna().unique())
count_order = sorted(pd.to_numeric(df[COL_GT]).dropna().astype(int).unique())
print(f"Loaded {len(df):,} predictions for {len(model_order)} models.")
print("Models:", model_order)

## 2. Exact Match by category and cardinality

In [ ]:
from data import COL_ANSWER, COL_COLOR, COL_GT, COL_MODEL, COL_SHAPE
from metrics import _exact_match_mask

CATEGORY_ORDER = [
    "cube",
    "sphere",
    "red",
    "green",
    "blue",
    "yellow",
    "purple",
]


def exact_match_by_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Exact Match rate per (model, category, cardinality).

    Categories are:
      - shape: cube, sphere
      - color: red, green, blue, yellow, purple

    Each example contributes once to its shape category
    and once to its color category.
    """
    pred = pd.to_numeric(df[COL_ANSWER], errors="coerce")
    gt = pd.to_numeric(df[COL_GT], errors="coerce")

    work = df.assign(
        _em=_exact_match_mask(pred, gt).astype(int)
    )

    # Shape categories
    shape = work[
        [COL_MODEL, COL_GT, COL_SHAPE, "_em"]
    ].rename(columns={COL_SHAPE: "category"})

    # Color categories
    color = work[
        [COL_MODEL, COL_GT, COL_COLOR, "_em"]
    ].rename(columns={COL_COLOR: "category"})

    # Each original example appears twice:
    # once for its shape and once for its color.
    categories = pd.concat(
        [shape, color],
        ignore_index=True,
    )

    long = (
        categories
        .groupby(
            [COL_MODEL, "category", COL_GT],
            as_index=False,
            observed=True,
        )["_em"]
        .mean()
        .rename(columns={"_em": "exact_match"})
    )

    return long

In [ ]:
em_category = exact_match_by_category(df)
em_category = em_category.sort_values([COL_MODEL, "category", COL_GT]).reset_index(drop=True)
em_category.to_csv(OUTPUT_DIR / "heatmap_accuracy_by_category.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'heatmap_accuracy_by_category.csv'}")

## 3. Heatmap (Figure 17)

In [ ]:
import matplotlib.ticker as mtick
def plot_category_accuracy_heatmap(
    df_long: pd.DataFrame,
    *,
    value: str = "exact_match",
    cmap: str = "Blues",
    vmin: float | None = 0.0,
    vmax: float | None = 1.0,
    cbar: bool = True,
    cbar_label: str = "Exact Match",
    model_order: Iterable[str] | None = None,
    count_order: Iterable[int] | None = None,
    save_to: str | Path | None = None,
) -> None:

    category_order = [
        "cube",
        "sphere",
        "red",
        "green",
        "blue",
        "yellow",
        "purple",
    ]

    d = df_long.copy()
    d[COL_GT] = d[COL_GT].astype(int)

    if count_order is None:
        count_order = sorted(d[COL_GT].unique())
    else:
        count_order = list(count_order)

    if model_order is None:
        model_order = sorted(d[COL_MODEL].unique())
    else:
        model_order = list(model_order)

    d["category"] = pd.Categorical(
        d["category"],
        categories=category_order,
        ordered=True,
    )

    pivot = d.pivot_table(
        index=[COL_MODEL, "category"],
        columns=COL_GT,
        values=value,
        aggfunc="mean",
        observed=False,
    )

    wanted_index = pd.MultiIndex.from_product(
        [model_order, category_order],
        names=[COL_MODEL, "category"],
    )

    pivot = pivot.reindex(
        index=wanted_index,
        columns=count_order,
    )

    n_models = len(model_order)
    n_categories = len(category_order)
    n_rows = len(pivot)

    cell_size = 0.22
    fig_w = max(7, len(count_order) * cell_size + 3)
    fig_h = max(8, n_rows * cell_size)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    hm = sns.heatmap(
        pivot,
        cmap=cmap,
        annot=False,
        linewidths=0.15,
        linecolor="white",
        cbar=cbar,
        cbar_kws={"aspect": 70},
        mask=pivot.isna(),
        square=True,
        vmin=vmin,
        vmax=vmax,
        ax=ax,
    )

    # ---------------------------------------------------------
    # Variantes : une étiquette pour CHAQUE ligne
    # ---------------------------------------------------------

    category_labels = [
        category
        for _, category in pivot.index
    ]

    # Important : on force une position de tick au centre
    # de chacune des 112 cellules/lignes.
    ax.set_yticks(
        np.arange(n_rows) + 0.5
    )

    ax.set_yticklabels(
        category_labels,
        rotation=0,
        fontsize=8,#7
    )

    ax.tick_params(
        axis="y",
        length=0,
        pad=3,
    )

    # ---------------------------------------------------------
    # Nom du modèle, une seule fois par groupe
    # ---------------------------------------------------------

    for i, model in enumerate(model_order):
        y_center = i * n_categories + n_categories / 2

        ax.text(
            -0.13,
            y_center,
            model,
            transform=ax.get_yaxis_transform(),
            ha="right",
            va="center",
            fontsize=8,
            fontweight="bold",
            clip_on=False,
        )

    # ---------------------------------------------------------
    # Séparation entre modèles
    # ---------------------------------------------------------

    for i in range(1, n_models):
        ax.axhline(
            i * n_categories,
            color="black",
            linewidth=1.5,
        )

    # ---------------------------------------------------------
    # Séparation formes / couleurs
    # ---------------------------------------------------------

    for i in range(n_models):
        ax.axhline(
            i * n_categories + 2,
            color="black",
            linewidth=0.4,
            alpha=0.6,
        )

    ax.set_xlabel(COL_GT)
    ax.set_ylabel("")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=0,
        fontsize=7,
    )

    if cbar and hm.collections and hm.collections[0].colorbar is not None:
        cb = hm.collections[0].colorbar
        cb.ax.yaxis.set_major_formatter(
            mtick.PercentFormatter(xmax=1.0)
        )
        cb.set_label(cbar_label)

    fig.subplots_adjust(left=0.25)

    _save_and_show(fig, save_to)

In [ ]:
plot_category_accuracy_heatmap(
    em_category,
    value="exact_match",
    cmap="Blues",
    model_order=model_order,
    count_order=count_order,
    save_to=OUTPUT_DIR / "heatmap_accuracy_by_category.pdf",
)

## 4. Pandas table grouped by shape and color (Table 6)

The table has the column hierarchy of Table 6. Each value is recalculated from the model predictions: Exact Match averaged over all videos in the given shape/color category, multiplied by 100 and rounded to the nearest integer. The two groups `Shape` and `Color` use the existing `metrics.exact_match_by_visual_attributes` function.

In [ ]:
attribute_em = exact_match_by_visual_attributes(df)

# Original metrics return rates in [0, 1]. Keep the requested header hierarchy.
shape_em = attribute_em["shape"].reindex(index=model_order, columns=["cube", "sphere"])
color_em = attribute_em["color"].reindex(
    index=model_order, columns=["red", "green", "blue", "yellow", "purple"]
)

category_table = pd.concat(
    [
        shape_em.rename(columns={"cube": "Cube", "sphere": "Sphere"}),
        color_em.rename(columns={
            "red": "Red", "green": "Green", "blue": "Blue",
            "yellow": "Yellow", "purple": "Purple",
        }),
    ],
    axis=1,
    keys=["Shape", "Color"],
)
category_table = (category_table * 100).round().astype("Int64")

# Place Model as the first visible column (rather than an extra CSV index row).
category_table.insert(0, ("Model", ""), category_table.index)
category_table = category_table.reset_index(drop=True)

display(category_table)
category_table.to_csv(OUTPUT_DIR / "table_accuracy_by_category.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'table_accuracy_by_category.csv'}")